# Model specification

## Multinomial logit model

Thermal sensation votes (TSV) were recorded on a seven-point ordinal scale ranging from $-3$ (cold) to $+3$ (hot), with $0$ representing the comfortable category. The data consist of 720 observations across 72 experimental conditions. Each observation represents the TSV counts of five participants of the same biological sex under a given combination of experimental conditions.

Because individual participant identifiers were unavailable, the model was fitted to aggregated category counts rather than individual-level repeated measurements. The response for observation $i$ is represented by a seven-dimensional count vector:

$$
\mathbf Y_i =
(Y_{i,-3},Y_{i,-2},Y_{i,-1},Y_{i,0},Y_{i,+1},Y_{i,+2},Y_{i,+3})
$$

where the counts sum to five:

$$
\sum_{k=-3}^{+3}Y_{i,k}=5
$$

The response vector is modelled using a multinomial likelihood:

$$
\mathbf Y_i
\sim
\operatorname{Multinomial}
\left(5,\mathbf p_i\right)
$$ {#eq-likelihood}

where $\mathbf p_i$ is the vector of probabilities for the seven TSV categories.

The comfortable category ($\mathrm{TSV}=0$) is used as the reference category. Its linear predictor is fixed to zero:

$$
\eta_{i,0}=0
$$

For the six non-reference categories, the category probabilities are obtained through the softmax transformation:

$$
p_{i,k}
=
\frac{\exp(\eta_{i,k})}
{1+\displaystyle\sum_{\ell\in\mathcal K}\exp(\eta_{i,\ell})},
\qquad
k\in\mathcal K
$$ {#eq-softmax}

where $\mathcal K=\{-3,-2,-1,+1,+2,+3\}$.

The probability of the comfortable category is consequently

$$
p_{i,0}
=
\frac{1}
{1+\displaystyle\sum_{\ell\in\mathcal K}
\exp(\eta_{i,\ell})}
$$ {#eq-reference-probability}

The linear predictor for each non-reference category $k$ is specified as

$$
\begin{aligned}
\eta_{i,k} =& \alpha_{\mathrm{condition}[i],\mathrm{sex}[i],k} + \beta_{\mathrm{condition}[i],\mathrm{t.onset},k}
\operatorname{mo}_k(\mathrm{t.onset}_i) + \beta_{\mathrm{t.day},k}\mathrm{t.day}_i \\
&+ \beta_{\mathrm{temp},k}\mathrm{temp}_i + \beta_{\mathrm{hum},k}\mathrm{hum}_i + \beta_{\mathrm{temp.hum},k}(\mathrm{temp}_i\mathrm{hum}_i)
\end{aligned}
$$ {#eq-linear-predictor}

Here, $k$ indexes the non-reference TSV categories, $\mathrm{sex}[i]$ identifies the biological sex of observation $i$, and $\mathrm{condition}[i]$ identifies its experimental condition. The variable $\mathrm{t.day}_{i}$ represents the time-of-day indicator (afternoon/evening), while $\mathrm{t.onset}_i$ denotes the ordered level of time since onset for observation $i$, and $\operatorname{mo}_k(\mathrm{t.onset}_i)$ represents its category-specific monotonic effect. The variable $\mathrm{temp}_{i}$ is centred air temperature, and $\mathrm{hum}_{i}$ is centred relative humidity expressed in 5% units. The temperature–humidity interaction is represented by their product.

## Multilevel structure

The sex coefficients and the monotonic time-since-onset coefficients are allowed to vary across experimental conditions. The condition-specific effects are decomposed into a constant (population-level) and condition-specific deviations:

$$
\alpha_{\mathrm{condition}[i],\mathrm{sex}[i],k} = \bar{\alpha}_{\mathrm{sex}[i],k} + v_{\mathrm{condition}[i],\mathrm{sex}[i],k}
$$ {#eq-sex-decomposition}

$$
\beta_{\mathrm{condition}[i],\mathrm{t.onset},k} = \bar{\beta}_{\mathrm{t.onset},k} + v_{\mathrm{condition}[i],\mathrm{t.onset},k}
$$ {#eq-onset-decomposition}

Here, $\bar{\alpha}_{\mathrm{sex}[i],k}$ and $\bar{\beta}_{\mathrm{t.onset},k}$ represent the constant (population-level) coefficients for the corresponding sex and monotonic time-since-onset effect, respectively. The terms $v_{\mathrm{condition}[i],\mathrm{sex}[i],k}$ and $v_{\mathrm{condition}[i],\mathrm{t.onset},k}$ represent condition-specific deviations from these constant (population-level) coefficients. This hierarchical structure allows partial pooling across experimental conditions.

The condition-varying effects are jointly modelled. For each condition $j$ (with $j=1,\ldots,72$ indexing the 72 experimental conditions), the vector of deviations is defined as

$$
\begin{aligned}
\mathbf v_j = (&
v_{j,\mathrm{male},-3},
v_{j,\mathrm{female},-3},
v_{j,\mathrm{t.onset},-3},
\\
&
v_{j,\mathrm{male},-2},
v_{j,\mathrm{female},-2},
v_{j,\mathrm{t.onset},-2},
\\
&
v_{j,\mathrm{male},-1},
v_{j,\mathrm{female},-1},
v_{j,\mathrm{t.onset},-1},
\\
&
v_{j,\mathrm{male},+1},
v_{j,\mathrm{female},+1},
v_{j,\mathrm{t.onset},+1},
\\
&
v_{j,\mathrm{male},+2},
v_{j,\mathrm{female},+2},
v_{j,\mathrm{t.onset},+2},
\\
&
v_{j,\mathrm{male},+3},
v_{j,\mathrm{female},+3},
v_{j,\mathrm{t.onset},+3}
)^\mathsf T
\end{aligned}
$$ {#eq-random-effect-vector}

Thus, each condition has a vector of 18 condition-varying effects, jointly modelled according to

$$
\mathbf v_j
\sim
\operatorname{MVN}
\left(
\mathbf 0_{18},\boldsymbol\Sigma_{18\times18}
\right),
\qquad j=1,\ldots,72
$$ {#eq-random-effects}

where $\boldsymbol\Sigma_{18\times18}$ is an $18\times18$ covariance matrix. This structure allows correlations between condition-varying effects across sexes, monotonic onset coefficients, and TSV categories.

### Non-centred parameterisation

For computational efficiency, the multivariate normal distribution of the condition-varying effects is implemented using a non-centred parameterisation. First, the standardised condition effects are defined as

$$
\mathbf z_j
\sim
\operatorname{MVN}
\left(
\mathbf 0,\mathbf I_{18}
\right),
\qquad j=1,\ldots,72
$$ {#eq-standard-normal-effects}

where $\mathbf I_{18}$ is the $18\times18$ identity matrix.

The condition-varying effects are then obtained through the transformation

$$
\mathbf v_j
=
\mathbf D_{\sigma}
\mathbf L_{\Omega}
\mathbf z_j,
\qquad j=1,\ldots,72
$$ {#eq-non-centred}

where $\mathbf D_{\sigma}$ is a diagonal matrix containing the group-level standard deviations, and $\mathbf L_{\Omega}$ is the Cholesky factor of the correlation matrix $\boldsymbol\Omega$.

The covariance matrix is therefore expressed as

$$
\boldsymbol\Sigma
=
\mathbf D_{\sigma}
\boldsymbol\Omega
\mathbf D_{\sigma},
\qquad
\boldsymbol\Omega
=
\mathbf L_{\Omega}
\mathbf L_{\Omega}^{\mathsf T}
$$ {#eq-covariance-decomposition}

This parameterisation ensures that the condition-varying effects follow the multivariate normal distribution specified in @eq-random-effects.

## Monotonic effect of time since onset

The effect of time since onset is modelled as a monotonic effect rather than assuming equal spacing between its ordered levels. Let $D$ denote the number of ordered levels of time since onset, and let $\mathrm{t.onset}_i\in\{1,\ldots,D\}$ denote the corresponding level for observation $i$. For each non-reference TSV category $k$, the monotonic effect is defined as

$$
\operatorname{mo}_k(\mathrm{t.onset}_i)
=
(D-1)
\sum_{d=1}^{D-1}
\zeta_{k,d}\mathbb I(\mathrm{t.onset}_i>d)
$$ {#eq-monotonic-effect}

where $\mathbb I(\cdot)$ is the indicator function and $\boldsymbol\zeta_k$ is a category-specific simplex vector.

The simplex weights satisfy

$$
\zeta_{k,d}\geq0,
\qquad
\sum_{d=1}^{D-1}\zeta_{k,d}=1
$$ {#eq-simplex-constraints}

The simplex weights determine how the overall monotonic effect is distributed across successive levels of time since onset. The scaling factor $D-1$ ensures that the monotonic effect ranges from zero at the first level to $D-1$ at the final level.

Each non-reference TSV category has its own simplex vector:

$$
\boldsymbol\zeta_k
\sim
\operatorname{Dirichlet}
(\boldsymbol\gamma_k),
\qquad
k\in\mathcal K
$$ {#eq-dirichlet-prior}

where $\boldsymbol\gamma_k$ denotes the vector of prior concentration parameters for category $k$.

The simplex weights are category-specific but are shared across experimental conditions. Condition-specific variation in the time-since-onset effect is captured by the corresponding condition-varying regression coefficient in @eq-onset-decomposition.

# Prior distributions

## Constant (population-level) regression coefficients

### Biological sex

The sex-specific baseline coefficients have category-dependent prior means:

$$
\begin{aligned}
\bar\alpha_{\mathrm{male}, -3},\bar\alpha_{\mathrm{female}, -3}
&\sim
\operatorname{Normal}(-2.69,2)\\
\bar\alpha_{\mathrm{male}, -2},\bar\alpha_{\mathrm{female}, -2}
&\sim
\operatorname{Normal}(-1.84,2)\\
\bar\alpha_{\mathrm{male}, -1},\bar\alpha_{\mathrm{female}, -1}
&\sim
\operatorname{Normal}(-0.89,2) \\
\bar\alpha_{\mathrm{male}, 1},\bar\alpha_{\mathrm{female}, 1}
&\sim
\operatorname{Normal}(-0.89,2) \\
\bar\alpha_{\mathrm{male}, 2},\bar\alpha_{\mathrm{female}, 2}
&\sim
\operatorname{Normal}(-1.84,2)\\
\bar\alpha_{\mathrm{male}, 3},\bar\alpha_{\mathrm{female}, 3}
&\sim
\operatorname{Normal}(-2.69,2)
\end{aligned}
$$

### Time since onset

The monotonic time since onset coefficients have the prior:

$$
\begin{aligned}
\bar\beta_{\mathrm{t.onset}, -3}
&\sim
\operatorname{Normal}(0,0.25)\\
\bar\beta_{\mathrm{t.onset}, -2}
&\sim
\operatorname{Normal}(0,0.25)\\
\bar\beta_{\mathrm{t.onset}, -1}
&\sim
\operatorname{Normal}(0,0.25)\\
\bar\beta_{\mathrm{t.onset}, 1}
&\sim
\operatorname{Normal}(0,0.25)\\
\bar\beta_{\mathrm{t.onset}, 2}
&\sim
\operatorname{Normal}(0,0.25)\\
\bar\beta_{\mathrm{t.onset}, 3}
&\sim
\operatorname{Normal}(0,0.25)
\end{aligned}
$$

Each non-reference TSV category has its own simplex parameter:

$$
\begin{aligned}
\boldsymbol{\zeta}_{-3}
\sim
\operatorname{Dirichlet}(2,2,2,2)\\
\boldsymbol{\zeta}_{-2}
\sim
\operatorname{Dirichlet}(2,2,2,2)\\
\boldsymbol{\zeta}_{-1}
\sim
\operatorname{Dirichlet}(2,2,2,2)\\
\boldsymbol{\zeta}_{1}
\sim
\operatorname{Dirichlet}(2,2,2,2)\\
\boldsymbol{\zeta}_{2}
\sim
\operatorname{Dirichlet}(2,2,2,2)\\
\boldsymbol{\zeta}_{3}
\sim
\operatorname{Dirichlet}(2,2,2,2)\\
\end{aligned}
$$

The Dirichlet prior is symmetric, assigning equal prior concentration to each increment of the monotonic effect.

### Time of day

The time-of-day coefficients have the priors:

$$
\begin{aligned}
\beta_{\mathrm{t.day}, -3}
&\sim
\operatorname{Normal}(0,0.25)\\
\beta_{\mathrm{t.day}, -2}
&\sim
\operatorname{Normal}(0,0.25)\\
\beta_{\mathrm{t.day}, -1}
&\sim
\operatorname{Normal}(0,0.25)\\
\beta_{\mathrm{t.day}, 1}
&\sim
\operatorname{Normal}(0,0.25)\\
\beta_{\mathrm{t.day}, 2}
&\sim
\operatorname{Normal}(0,0.25)\\
\beta_{\mathrm{t.day}, 3}
&\sim
\operatorname{Normal}(0,0.25)
\end{aligned}
$$

### Air temperature

The air temperature coefficients have the priors:

$$
\begin{aligned}
\beta_{\mathrm{temp}, -3}
&\sim
\operatorname{Normal}(-1,0.5)\\
\beta_{\mathrm{temp}, -2}
&\sim
\operatorname{Normal}(-1,0.5)\\
\beta_{\mathrm{temp}, -1}
&\sim
\operatorname{Normal}(-1,0.5)\\
\beta_{\mathrm{temp}, 1}
&\sim
\operatorname{Normal}(1,0.5)\\
\beta_{\mathrm{temp}, 2}
&\sim
\operatorname{Normal}(1,0.5)\\
\beta_{\mathrm{temp}, 3}
&\sim
\operatorname{Normal}(1,0.5)
\end{aligned}
$$

### Relative humidity

The relative humidity coefficients have the priors:

$$
\begin{aligned}
\beta_{\mathrm{hum}, -3}
&\sim
\operatorname{Normal}(-0.5,0.25)\\
\beta_{\mathrm{hum}, -2}
&\sim
\operatorname{Normal}(-0.5,0.25)\\
\beta_{\mathrm{hum}, -1}
&\sim
\operatorname{Normal}(-0.5,0.25)\\
\beta_{\mathrm{hum}, 1}
&\sim
\operatorname{Normal}(0.5,0.25)\\
\beta_{\mathrm{hum}, 2}
&\sim
\operatorname{Normal}(0.5,0.25)\\
\beta_{\mathrm{hum}, 3}
&\sim
\operatorname{Normal}(0.5,0.25)
\end{aligned}
$$

### Air temperature and relative humidity interaction

The air temperature and relative humidity interaction coefficients have the priors:

$$
\begin{aligned}
\beta_{\mathrm{temp.hum}, -3}
&\sim
\operatorname{Normal}(0,0.2)\\
\beta_{\mathrm{temp.hum}, -2}
&\sim
\operatorname{Normal}(0,0.2)\\
\beta_{\mathrm{temp.hum}, -1}
&\sim
\operatorname{Normal}(0,0.2)\\
\beta_{\mathrm{temp.hum}, 1}
&\sim
\operatorname{Normal}(0,0.2)\\
\beta_{\mathrm{temp.hum}, 2}
&\sim
\operatorname{Normal}(0,0.2)\\
\beta_{\mathrm{temp.hum}, 3}
&\sim
\operatorname{Normal}(0,0.2)
\end{aligned}
$$

## Condition-varying effects

The standard deviations of the condition-varying effects are assigned exponential priors:

$$
\sigma_r
\sim
\operatorname{Exponential}(1),
\qquad r=1,\ldots,18
$$ {#eq-sd-priors}

where the index $r$ identifies the 18 dimensions of the condition-varying effects vector.

The correlation matrix is assigned an LKJ prior:

$$
\boldsymbol\Omega
\sim
\operatorname{LKJ}(2)
$$ {#eq-lkj-prior}

The standardised effects used in the non-centred parameterisation have independent standard normal priors:

$$
z_{j,r}
\sim
\operatorname{Normal}(0,1),
\qquad j=1,\ldots,72,
\qquad r=1,\ldots,18
$$ {#eq-z-prior}